### Hypothesis Testing 
In this notebook we will utilize statistical learning methods to determine whether or not the input variables (X) are effective in explaining the predictor (Y). To do this, we must compute the standard errors of the coeficent for each input and T-test it to determine if the coeficent is statisticaly significant. Our chosen model for this phase will be K-Nearest Neighbors and Logistic Regression. These simple models outputted the best evalution results out of all the different algorithms that we're fitted and tested. 

In [11]:
# import data 
import pandas as pd 
from pathlib import Path 

# load directory 
dir = Path.cwd()

# point to the data opath file 
X_train = dir.parent / 'Data' / 'X_train.csv'
Y_train = dir.parent / 'Data'/ 'Y_train.csv'

# load data 
X_train = pd.read_csv(X_train)
Y_train = pd.read_csv(Y_train)

# make data 1D (data will load with index if not ignored)
Y_train = Y_train['Survived'].values.ravel() 


# K-Nearest Neighbors
Important Insight: Since KNN does not have p-values because it is a non-parametric model, we will use sequential feature selection which automatically handles the backward elimintation and determines the best input features by checking accuaracy/precision at each step. 


In [12]:
from sklearn.feature_selection import SequentialFeatureSelector # elimination process
from sklearn.neighbors import KNeighborsClassifier # model 

# initialze model with the gridsearch adjusted parameters 
knn = KNeighborsClassifier(leaf_size=10, weights='distance')

# backward elimination for KKN 
sfs = SequentialFeatureSelector(knn, 
                                n_features_to_select = 'auto', 
                                direction = 'backward', 
                                scoring = 'precision')
# fit data onto model 
sfs.fit(X_train, Y_train)

# best features
selected_features = X_train.columns[sfs.get_support()]

# results
selected_features

C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result)

Index(['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction7', 'Ratio4'],
      dtype='object')

## Logistic Regression 
Important Insight: Redundancy is prevelant between the interaction variables, causing the model to suffer when predicting the raw column fare which is made up of the engieneered features. Also, removing the redundant categories like male and embarked to allow the model to use one category as the baseline. 

In [13]:
import statsmodels.api as sm 

#convert bool objects to floats (statsmodel has a hard time seeing them as bool objects)
X_train_numeric = X_train.astype(float)
                    
# manually add a constant intercept to the data 
X_train_const = sm.add_constant(X_train_numeric)

# fit the model
model = sm.Logit(Y_train, X_train_const)
result = model.fit(maxiter = 1000)

# view results 
print(f'Summary {result.summary()}')


         Current function value: 0.434132
         Iterations: 1000
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      605
Method:                           MLE   Df Model:                           17
Date:                Thu, 16 Apr 2026   Pseudo R-squ.:                  0.3416
Time:                        16:27:43   Log-Likelihood:                -270.46
converged:                      False   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 1.090e-49
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.6335      0.610     -1.038      0.299      -1.829       0.562
Unnamed: 0       -0.0003      0.001     -0.514

C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [15]:
# variance inflation factor
from statsmodels.stats.outliers_influence import variance_inflation_factor

# intercept has to be constant, but we did this already upstream 

# calculate VIF for each feature 
vif_data = pd.DataFrame()
vif_data['feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i)]
print(vif_data.sort_values(by = 'VIF', ascending = False))

NameError: name 'i' is not defined

In [ ]:
X_train_const.corr().abs()

,const,Unnamed: 0,Pclass,Age,SibSp,Parch,Fare,Embarked_644,Embarked_C,Embarked_Q,...,Sex_female,Sex_male,Cabin_encoded,interaction4,interaction5,interaction6,interaction7,Ratio1,Ratio4,Ratio5
const,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unnamed: 0,NaN,1.000000,0.021330,0.023622,0.000066,0.089435,0.031785,0.054177,0.052572,0.024570,...,0.014864,0.014864,0.044673,0.026004,0.045073,0.020137,0.025194,0.008774,0.016849,0.003176
Pclass,NaN,0.021330,1.000000,0.322295,0.097146,0.030596,0.640704,0.065673,0.218929,0.222335,...,0.127310,0.127310,0.377057,0.191727,0.538481,0.494409,0.440527,0.039737,0.086893,0.120847
Age,NaN,0.023622,0.322295,1.000000,0.231572,0.178476,0.081839,0.027091,0.003773,0.041854,...,0.089229,0.089229,0.099918,0.295577,0.016024,0.090580,0.035405,0.178707,0.029352,0.081154
SibSp,NaN,0.000066,0.097146,0.231572,1.000000,0.440442,0.328730,0.019012,0.081010,0.007884,...,0.092611,0.092611,0.003097,0.140524,0.478863,0.149920,0.304948,0.079833,0.049968,0.082740
Parch,NaN,0.089435,0.030596,0.178476,0.440442,1.000000,0.336510,0.019219,0.034177,0.100082,...,0.258301,0.258301,0.031448,0.035491,0.430356,0.219736,0.261754,0.089787,0.196252,0.091857
Fare,NaN,0.031785,0.640704,0.081839,0.328730,0.336510,1.000000,0.059984,0.216111,0.156978,...,0.247446,0.247446,0.289299,0.163577,0.923790,0.620491,0.802568,0.004290,0.078103,0.150656
Embarked_644,NaN,0.054177,0.065673,0.027091,0.019012,0.019219,0.059984,1.000000,0.018567,0.012477,...,0.055630,0.055630,0.143154,0.038003,0.041813,0.092183,0.006330,0.004924,0.006753,0.000528
Embarked_C,NaN,0.052572,0.218929,0.003773,0.081010,0.034177,0.216111,0.018567,1.000000,0.144094,...,0.083348,0.083348,0.038232,0.119290,0.135308,0.163714,0.150912,0.024903,0.049753,0.077438
Embarked_Q,NaN,0.024570,0.222335,0.041854,0.007884,0.100082,0.156978,0.012477,0.144094,1.000000,...,0.061973,0.061973,0.042746,0.022659,0.165815,0.210002,0.040331,0.007689,0.048574,0.029601
